In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from math import nan
import pandas as pd
import yaml
import xesmf as xe
import dask
import socket
from pathlib import Path
import sys

sys.path.append("/glade/u/home/islas/python/CESM2_with_CMIP7_aer/utils/")
from reading_utils import *
from convert_units import *
from spread_emissions_func import *
from molecular_weights import *
from vertical_utils import *
from dask_utils import *

import os
import csv

from CASutils import averaging_utils as avg
import dask
import socket
from datetime import datetime

#### Set up dask cluster

In [2]:
cluster = get_dask_cluster(8)

/glade/work/islas/conda-envs/islaenv26/lib/python3.10/site-packages/dask_jobqueue/core.py:266: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/glade/work/islas/conda-envs/islaenv26/lib/python3.10/site-packages/dask_jobqueue/pbs.py:82: FutureWarning: project has been renamed to account as this kwarg was used wit -A option. You are still using it (please also check config files). If you did not set account yet, project will be respected for now, but it will be removed in a future release. If you already set account, project is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/glade/work/islas/conda-envs/islaenv26/lib/python3.1

In [5]:
cluster

PBSCluster(ec5fccd7, 'tcp://10.18.206.70:38521', workers=8, threads=8, memory=223.52 GiB)

#### Set up species info an log file

In [6]:
process_species = ['so4_a1']
emiss_type = "elev"
forcing_type = "anthro"

for species in process_species:
    print(species)

    # Set up output path and log file
    outpath="/glade/campaign/cgd/cas/islas/python_savs/CESM2_with_CMIP7_aer/make_emissions/"+forcing_type+"/"+emiss_type+"/CMIP7/"
    os.makedirs(outpath, exist_ok=True)
    os.makedirs(outpath+'/DIAGS/', exist_ok=True)
    logfilename = outpath+'/DIAGS/'+species+'_log.csv'
    if os.path.exists(logfilename):
        os.remove(logfilename)

    # open log file for writing
    file = open(logfilename, mode='w', newline='')
    writer = csv.writer(file)

    # Set up output filename
    fout = outpath+"/"+species+"_"+forcing_type+"_"+emiss_type+"_cmip7_for_cesm2_negspread.nc"

    # Set up spreading diagnostic filename
    fout_diags = outpath+"/DIAGS/DIAGS_"+species+"_"+forcing_type+"_"+emiss_type+"_cmip7_for_cesm2_negspread_diags.nc"

    # Read in the cmip6 piControl, cmip7 piControl, cmip6 historical and cmip7 historical
    cmip6_pi = read_emissions('../../../../emission_file_'+forcing_type+'_'+emiss_type+'_info.yml',species,'cmip6','piControl')
    cmip7_pi = read_emissions('../../../../emission_file_'+forcing_type+'_'+emiss_type+'_info.yml',species,'cmip7','piControl', dstgrid = 'f09')
    cmip6_hist = read_emissions('../../../../emission_file_'+forcing_type+'_'+emiss_type+'_info.yml',species,'cmip6','hist')
    cmip7_hist = read_emissions('../../../../emission_file_'+forcing_type+'_'+emiss_type+'_info.yml',species,'cmip7','hist', dstgrid='f09')

    # save the attributes for CMIP7
    cmip7_attrs = cmip7_hist.attrs

    # Enforce the same longitudes and latitudes for all arrays
    cmip7_pi['lon'] = cmip6_pi.lon ; cmip7_pi['lat'] = cmip6_pi.lat
    cmip6_hist['lon'] = cmip6_pi.lon ; cmip6_hist['lat'] = cmip6_pi.lat
    cmip7_hist['lon'] = cmip6_pi.lon ; cmip7_hist['lat'] = cmip6_pi.lat

    # Sum up the different emission types
    cmip6_pi = cmip6_pi.sum('emiss_type')
    cmip7_pi = cmip7_pi.sum('emiss_type')
    cmip6_hist = cmip6_hist.sum('emiss_type')
    cmip7_hist = cmip7_hist.sum('emiss_type')

    cmip6_pi = cmip6_pi.chunk({'time':120})
    cmip6_hist = cmip6_hist.chunk({'time':120})
    cmip7_pi = cmip7_pi.chunk({'time':120})
    cmip7_hist = cmip7_hist.chunk({'time':120})

    # Vertically integrate the elevated emissions
    cmip6_pi_column = vertically_integrate(cmip6_pi)
    cmip7_pi_column = vertically_integrate(cmip7_pi)
    cmip6_hist_column = vertically_integrate(cmip6_hist)
    cmip7_hist_column = vertically_integrate(cmip7_hist)

    # Add the CMIP7 anomalies onto the CMIP6 piControl anomalies (For the vertical integral)
    cmip7_1850 = cmip7_pi_column.groupby('time.month').mean('time')
    cmip6_1850 = cmip6_pi_column.groupby('time.month').mean('time')
    cmip7_anom = cmip7_hist_column.groupby('time.month') - cmip7_1850
    cmip7_for_cesm2 = cmip6_1850 + cmip7_anom.groupby('time.month')

    # ditch the pre-1850 years
    cmip7_for_cesm2 = cmip7_for_cesm2.sel(time=slice("1850-01-01","2100-12-31"))

    cmip7_for_cesm2 = cmip7_for_cesm2.load()

    nring=60
    output_spread_by_ring_tg = []

    writer.writerow(['time','lon','lat','deficit'])
    lon = cmip7_for_cesm2.lon.values
    lat = cmip7_for_cesm2.lat.values
    fixed = np.empty( cmip7_for_cesm2.shape, dtype = cmip7_for_cesm2.dtype)
    for itime in np.arange(0,cmip7_for_cesm2.time.size):

        this_time = cmip7_for_cesm2.isel(time=itime).load()

        if itime % 120 == 0:
             print(this_time.time.values)
        #    print( "memory=",process.memory_info().rss / 1e9, "GB" )

        newdat = this_time.values.copy()
        ilat_neg, ilon_neg = np.where( newdat < 0)

        # Note "time" is really "ring" here
        spread_by_ring = xr.DataArray(np.zeros((nring, len(lat), len(lon)), dtype=np.float64), dims=['time','lat','lon'],
                                  coords=[np.arange(0,60,1), cmip7_for_cesm2.lat, cmip7_for_cesm2.lon])
    
        for ilat, ilon in zip(ilat_neg, ilon_neg):
            writer.writerow([this_time.time.values, lon[ilon], lat[ilat], newdat[ilat,ilon]])
            newdat, this_spread = spread_negative_emissions_np(newdat,lon,lat,ilon,ilat)
            spread_by_ring[:,ilat,ilon] += this_spread

        # Convert spread by ring to Tg
        spread_by_ring_tg = convert_molecules_to_tg_specifywgt(spread_by_ring, mol_weights[species])
        spread_by_ring_tg = spread_by_ring_tg.rename({'time':'ring'})
        output_spread_by_ring_tg.append(spread_by_ring_tg)

        fixed[itime,:,:] = newdat

    fixed = xr.DataArray(fixed, dims=cmip7_for_cesm2.dims, coords = cmip7_for_cesm2.coords, name='emiss')
    fixed_vertical = vertically_distribute_as_cmip6(fixed, cmip6_pi.altitude.values)
    fixed_vertical.attrs = cmip7_attrs
    fixed_vertical.attrs["description"] = "CMIP7 emission anomalies added onto the CMIP6 piControl, adjusted to spread negative values"
    fixed_vertical.attrs["method"] = "Negative emissions redistributed to neighboring grid cells"

    # add date variable
    date = [str(itime.dt.year.values).zfill(4)+str(itime.dt.month.values).zfill(2)+str(itime.dt.day.values).zfill(2) for itime in fixed_vertical.time ]
    date = xr.DataArray(date, dims=['time'], coords=[fixed.time], name='date')
    date.attrs['units'] = 'YYYYMMDD'
    date.attrs['long_name'] = 'Date'

    datout = xr.merge([fixed, date])
    datout.attrs.update({
        "author": "Isla Simpson (islas@ucar.edu)",
        "creation_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "creation_script": os.path.basename("/glade/u/home/islas/python/CESM2_with_CMIP7_aer/make_emissions/anthro/sfc/CMIP7/make_anthro_emissions.ipynb")})
    datout.to_netcdf(fout)

    output_spread_by_ring_tg = xr.concat(output_spread_by_ring_tg, dim='time')
    output_spread_by_ring_tg['time'] = fixed.time.values
    output_spread_by_ring_tg.to_netcdf(fout_diags)
    file.close()

cluster.close()

so4_a1
/glade/p/cesmdata/cseg/inputdata/atm/cam/chem/emis/CMIP6_emissions_1750_2015/emissions-cmip6_so4_a1_anthro-ene_vertical_1750-2015_0.9x1.25_c20170616.nc
emiss_ene_ind
/glade/campaign/cgd/cas/islas/python_savs/CESM2_with_CMIP7_aer/make_emissions/REMAP/f09/so4_a1_ene_vertical-em-anthro_input4MIPs_emissions_CMIP_CEDS-CMIP-2025-04-18_gn_175001-202312_c20251030.nc
emiss
/glade/p/cesmdata/cseg/inputdata/atm/cam/chem/emis/CMIP6_emissions_1750_2015/emissions-cmip6_so4_a1_anthro-ene_vertical_1750-2015_0.9x1.25_c20170616.nc
emiss_ene_ind
/glade/campaign/cgd/cas/islas/python_savs/CESM2_with_CMIP7_aer/make_emissions/REMAP/f09/so4_a1_ene_vertical-em-anthro_input4MIPs_emissions_CMIP_CEDS-CMIP-2025-04-18_gn_175001-202312_c20251030.nc
emiss


/glade/work/islas/conda-envs/islaenv26/lib/python3.10/site-packages/distributed/client.py:3398: UserWarning: Sending large graph of size 5.46 GiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


1850-01-16 00:00:00
1860-01-16 00:00:00
1870-01-16 00:00:00
1880-01-16 00:00:00
1890-01-16 00:00:00
1900-01-16 00:00:00
1910-01-16 00:00:00
1920-01-16 00:00:00
1930-01-16 00:00:00
1940-01-16 00:00:00
1950-01-16 00:00:00
1960-01-16 00:00:00
1970-01-16 00:00:00
1980-01-16 00:00:00
1990-01-16 00:00:00
2000-01-16 00:00:00
2010-01-16 00:00:00
2020-01-16 00:00:00


/glade/work/islas/conda-envs/islaenv26/lib/python3.10/site-packages/dask_jobqueue/core.py:266: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/glade/work/islas/conda-envs/islaenv26/lib/python3.10/site-packages/dask_jobqueue/pbs.py:82: FutureWarning: project has been renamed to account as this kwarg was used wit -A option. You are still using it (please also check config files). If you did not set account yet, project will be respected for now, but it will be removed in a future release. If you already set account, project is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
